# SmartGrid Replay Benchmark Analyse

Ce notebook sert a piloter et analyser un **replay multi-jours**.

Workflow :
1. choisir une plage de dates ;
2. choisir les modeles a comparer ;
3. lancer le replay benchmark directement depuis le notebook si besoin ;
4. charger les sorties et comparer baseline, current et autres runs.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
plt.style.use('seaborn-v0_8')


## 1. Configuration utilisateur

Si `RUN_REPLAY_BENCHMARK = True`, le notebook lance `scripts/benchmark_replay_models.py` sur la plage demandee.
Sinon, il recharge un benchmark replay existant.


In [ ]:
MANUAL_PROJECT_ROOT = None

DATASET_KEY = 'clean_v1'
DATASET_OVERRIDES = {
    'historical_csv': None,
    'forecast_csv': None,
    'benchmark_csv': None,
    'weather_csv': None,
    'holidays_xlsx': None,
}

RUN_REPLAY_BENCHMARK = False
START_DATE = '2025-11-01'
END_DATE = '2025-11-07'
EXECUTION_DEVICE = 'auto'
ALLOW_FALLBACK = False

ARTIFACTS_ROOT = None

MANUAL_REPLAY_SUMMARY_CSV = None
MANUAL_REPLAY_MANIFEST_JSON = None

BASELINE_CONFIG_KEYWORD = 'mlp_baseline'
BASELINE_RUN_ID = None
CURRENT_RUN_ID = None
COMPARE_WITH_BASELINE = True
COMPARE_WITH_CURRENT = True
TOP_N_OTHER_MODELS = 3
EXTRA_MODEL_RUN_IDS = []
REQUESTED_MODEL_RUN_IDS = None
SELECTED_MODELS = None
SELECTED_DAY = None

REFERENCE_SORT_METRIC = 'RMSE'
PLOT_LAST_N_POINTS = 7 * 144
DISPLAY_TOP_ROWS = 12


## 2. Fonctions utilitaires

Cette cellule reconstruit le catalogue des runs, resout baseline/current, permet de lancer le replay benchmark, puis de charger les sorties multi-modeles.


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'smartgrid').exists():
            return candidate
    raise FileNotFoundError('Impossible de trouver la racine du projet.')


def ensure_project_on_path(project_root: Path) -> None:
    src_dir = project_root / 'src'
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))


def first_existing(paths) -> Path | None:
    for path in paths:
        if path and Path(path).exists():
            return Path(path)
    return None


def load_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'))


def resolve_notebook_data_config(project_root: Path) -> dict:
    from smartgrid.data.catalog import resolve_consumption_data_config

    base_cfg = {'dataset_key': DATASET_KEY} if DATASET_KEY else {}
    return resolve_consumption_data_config(
        base_cfg,
        project_root=project_root,
        overrides=dict(DATASET_OVERRIDES or {}),
    )


def resolve_path(path_like, project_root: Path) -> Path | None:
    if not path_like:
        return None
    raw = Path(path_like).expanduser()
    candidates = [raw]
    if not raw.is_absolute():
        candidates.append(project_root / raw)
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return (project_root / raw).resolve() if not raw.is_absolute() else raw.resolve()


def find_feature_benchmark_csv(project_root: Path) -> Path | None:
    return first_existing([
        project_root / 'artifacts' / 'benchmarks' / 'consumption_feature_variants.csv',
        project_root / 'artifacts' / 'benchmarks' / 'feature_variants.csv',
    ])


def find_latest_replay_benchmark(project_root: Path) -> tuple[Path, Path | None]:
    base = project_root / 'artifacts' / 'benchmarks' / 'replay'
    if not base.exists():
        raise FileNotFoundError('Aucun dossier artifacts/benchmarks/replay trouve.')
    candidates = sorted(path for path in base.iterdir() if path.is_dir())
    if not candidates:
        raise FileNotFoundError('Aucun benchmark replay disponible.')
    latest = candidates[-1]
    summary_csv = latest / 'replay_benchmark_summary.csv'
    manifest_json = latest / 'replay_benchmark_manifest.json'
    return summary_csv.resolve(), manifest_json.resolve() if manifest_json.exists() else None


def format_feature_signature(feature_cfg: dict | None) -> str:
    feature_cfg = dict(feature_cfg or {})
    parts = []
    if feature_cfg.get('include_calendar', True):
        parts.append('cal')
    if feature_cfg.get('include_temperature', False):
        parts.append('temp')
    if feature_cfg.get('include_manual_daily_lags', False):
        parts.append('lags')
    if feature_cfg.get('include_cyclical_time', False):
        parts.append('cyc')
    if feature_cfg.get('include_lag_aggregates', False):
        parts.append('lagagg')
    if feature_cfg.get('include_recent_dynamics', False):
        parts.append('dyn')
    if feature_cfg.get('include_weather', False):
        parts.append(f"weather={feature_cfg.get('weather_mode', 'custom')}")
    if not feature_cfg.get('include_temperature', False) and not feature_cfg.get('include_weather', False):
        parts.append('no-weather')
    lag_days = feature_cfg.get('lag_days') or []
    if lag_days:
        parts.append('J=' + '-'.join(str(x) for x in lag_days))
    return ' | '.join(parts) if parts else 'features inconnues'


def collect_runs(project_root: Path) -> pd.DataFrame:
    rows = []
    runs_dir = project_root / 'artifacts' / 'runs' / 'consumption'
    for run_summary_path in sorted(runs_dir.glob('*/run_summary.json')):
        payload = load_json(run_summary_path)
        config_path = payload.get('config_path')
        config_name = Path(config_path).name if config_path else None
        metrics_model = payload.get('metrics_model') or {}
        rows.append({
            'run_id': str(payload.get('run_id', run_summary_path.parent.name)),
            'run_summary_path': str(run_summary_path.resolve()),
            'bundle_dir': str(run_summary_path.parent.resolve()),
            'config_path': config_path,
            'config_name': config_name,
            'experiment_name': payload.get('experiment_name'),
            'feature_config': payload.get('feature_config') or {},
            'feature_columns': payload.get('feature_columns') or [],
            'hidden_layers': payload.get('hidden_layers') or [],
            'n_features': payload.get('n_features'),
            'historical_csv': payload.get('historical_csv'),
            'weather_csv': payload.get('weather_csv'),
            'holidays_xlsx': payload.get('holidays_xlsx'),
            'MAE_train_benchmark': metrics_model.get('MAE'),
            'RMSE_train_benchmark': metrics_model.get('RMSE'),
        })
    catalog_df = pd.DataFrame(rows)
    if catalog_df.empty:
        return catalog_df
    catalog_df['feature_signature'] = catalog_df['feature_config'].map(format_feature_signature)
    catalog_df['arch_label'] = catalog_df['hidden_layers'].map(lambda value: f'MLP {value}' if isinstance(value, list) and value else 'MLP')
    catalog_df['short_name'] = catalog_df.apply(
        lambda row: Path(row['config_name']).stem if pd.notna(row.get('config_name')) and row.get('config_name') else (row.get('experiment_name') or row['run_id']),
        axis=1,
    )
    catalog_df['display_name'] = catalog_df['short_name'] + ' | ' + catalog_df['feature_signature']
    catalog_df['is_baseline_config'] = catalog_df['config_name'].fillna('').str.contains(BASELINE_CONFIG_KEYWORD, case=False, na=False)
    return catalog_df.sort_values(['config_name', 'run_id']).reset_index(drop=True)


def current_run_id(project_root: Path) -> str | None:
    path = project_root / 'artifacts' / 'models' / 'consumption' / 'current' / 'run_summary.json'
    if not path.exists():
        return None
    run_id = str(load_json(path).get('run_id', '')).strip()
    return run_id or None


def load_feature_benchmark(feature_benchmark_csv: Path | None) -> pd.DataFrame:
    if feature_benchmark_csv is None or not feature_benchmark_csv.exists():
        return pd.DataFrame()
    bench = pd.read_csv(feature_benchmark_csv).copy()
    bench.columns = [str(col).strip() for col in bench.columns]
    if 'config' in bench.columns and 'config_name' not in bench.columns:
        bench = bench.rename(columns={'config': 'config_name'})
    return bench if 'run_id' in bench.columns else pd.DataFrame()


def resolve_baseline_run_id(catalog_df: pd.DataFrame, feature_benchmark_df: pd.DataFrame) -> str | None:
    if BASELINE_RUN_ID:
        return str(BASELINE_RUN_ID)
    if not feature_benchmark_df.empty and 'config_name' in feature_benchmark_df.columns:
        candidates = feature_benchmark_df[
            feature_benchmark_df['config_name'].astype(str).str.contains(BASELINE_CONFIG_KEYWORD, case=False, na=False)
        ].copy()
        sort_cols = [col for col in [REFERENCE_SORT_METRIC, 'RMSE', 'MAE'] if col in candidates.columns]
        if not candidates.empty:
            return str(candidates.sort_values(sort_cols or ['run_id']).iloc[0]['run_id'])
    if not catalog_df.empty:
        candidates = catalog_df[catalog_df['is_baseline_config']].copy()
        if not candidates.empty:
            sort_cols = [col for col in ['RMSE_train_benchmark', 'MAE_train_benchmark', 'run_id'] if col in candidates.columns]
            return str(candidates.sort_values(sort_cols, ascending=True).iloc[0]['run_id'])
    return None


def resolve_inputs(project_root: Path, catalog_df: pd.DataFrame, current_id: str | None, baseline_id: str | None):
    data_cfg = resolve_notebook_data_config(project_root)
    historical_csv = resolve_path(data_cfg.get('historical_csv'), project_root)
    weather_csv = resolve_path(data_cfg.get('weather_csv'), project_root)
    holidays_xlsx = resolve_path(data_cfg.get('holidays_xlsx'), project_root)
    artifacts_root = resolve_path(ARTIFACTS_ROOT or (project_root / 'artifacts'), project_root)
    return historical_csv, weather_csv, holidays_xlsx, artifacts_root


def default_requested_models(catalog_df: pd.DataFrame, feature_benchmark_df: pd.DataFrame, current_id: str | None, baseline_id: str | None) -> list[str]:
    if REQUESTED_MODEL_RUN_IDS:
        return list(dict.fromkeys(str(run_id) for run_id in REQUESTED_MODEL_RUN_IDS))
    selected = []
    excluded = set()
    if COMPARE_WITH_CURRENT and current_id:
        selected.append(str(current_id))
        excluded.add(str(current_id))
    if COMPARE_WITH_BASELINE and baseline_id:
        selected.append(str(baseline_id))
        excluded.add(str(baseline_id))

    other_count = 0
    if not feature_benchmark_df.empty:
        sort_cols = [col for col in [REFERENCE_SORT_METRIC, 'RMSE', 'MAE'] if col in feature_benchmark_df.columns]
        for run_id in feature_benchmark_df.sort_values(sort_cols or ['run_id'])['run_id'].astype(str):
            if run_id in excluded:
                continue
            selected.append(run_id)
            excluded.add(run_id)
            other_count += 1
            if other_count >= TOP_N_OTHER_MODELS:
                break

    if other_count < TOP_N_OTHER_MODELS and not catalog_df.empty:
        ordered = catalog_df.sort_values([col for col in ['RMSE_train_benchmark', 'MAE_train_benchmark', 'run_id'] if col in catalog_df.columns], ascending=True)
        for run_id in ordered['run_id'].astype(str):
            if run_id in excluded:
                continue
            selected.append(run_id)
            excluded.add(run_id)
            other_count += 1
            if other_count >= TOP_N_OTHER_MODELS:
                break

    for run_id in EXTRA_MODEL_RUN_IDS:
        run_id = str(run_id)
        if run_id not in excluded:
            selected.append(run_id)
            excluded.add(run_id)
    return selected


def build_metadata(summary_df: pd.DataFrame, catalog_df: pd.DataFrame) -> pd.DataFrame:
    if summary_df.empty or catalog_df.empty:
        return pd.DataFrame()
    meta = catalog_df.rename(columns={'run_id': 'requested_model_run_id'}).drop_duplicates(subset=['requested_model_run_id']).copy()
    keep = [
        'requested_model_run_id', 'run_summary_path', 'bundle_dir', 'config_path', 'config_name',
        'experiment_name', 'feature_config', 'feature_columns', 'hidden_layers', 'n_features',
        'historical_csv', 'weather_csv', 'holidays_xlsx', 'feature_signature', 'arch_label',
        'short_name', 'display_name', 'is_baseline_config', 'MAE_train_benchmark', 'RMSE_train_benchmark'
    ]
    keep = [col for col in keep if col in meta.columns]
    return meta[keep]


def add_labels(df: pd.DataFrame, metadata_df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or metadata_df.empty:
        return df.copy()
    extra_cols = [col for col in metadata_df.columns if col != 'requested_model_run_id' and col not in df.columns]
    enriched = df.merge(metadata_df[['requested_model_run_id'] + extra_cols], on='requested_model_run_id', how='left')
    counts = enriched.groupby('display_name')['requested_model_run_id'].transform('nunique') if 'display_name' in enriched.columns else 0
    suffix = enriched['requested_model_run_id'].astype(str).str.replace('consumption_mlp_', '', regex=False)
    if 'display_name' in enriched.columns:
        enriched['model_label'] = np.where(counts > 1, enriched['display_name'] + ' [' + suffix + ']', enriched['display_name'])
    return enriched


def compute_metrics(real: pd.Series, pred: pd.Series) -> dict:
    mask = real.notna() & pred.notna()
    y = real[mask].astype(float)
    yhat = pred[mask].astype(float)
    if len(y) == 0:
        return {'count': 0}
    err = yhat - y
    abs_err = err.abs()
    ape = abs_err / y.replace(0, np.nan).abs()
    smape = abs_err / ((y.abs() + yhat.abs()) / 2).replace(0, np.nan)
    tol = np.maximum(0.01 * y.abs(), 5000.0)
    ramp = (yhat.diff() - y.diff()).dropna()
    return {
        'count': int(mask.sum()),
        'MAE': float(abs_err.mean()),
        'RMSE': float(np.sqrt((err ** 2).mean())),
        'Bias(ME)': float(err.mean()),
        'MAPE%': float(100 * ape.dropna().mean()) if ape.dropna().size else np.nan,
        'SMAPE%': float(100 * smape.dropna().mean()) if smape.dropna().size else np.nan,
        'InTolerance%': float(100 * (abs_err <= tol).mean()),
        'P95AbsError': float(abs_err.quantile(0.95)),
        'P99AbsError': float(abs_err.quantile(0.99)),
        'UnderShare%': float(100 * (err < 0).mean()),
        'OverShare%': float(100 * (err > 0).mean()),
        'Under_MAE': float(abs_err[err < 0].mean()) if (err < 0).any() else np.nan,
        'Over_MAE': float(abs_err[err > 0].mean()) if (err > 0).any() else np.nan,
        'CorrAbsErr_vs_Real': float(abs_err.corr(y)) if len(y) > 1 else np.nan,
        'RampingError_RMSE': float(np.sqrt((ramp ** 2).mean())) if len(ramp) else np.nan,
    }


def delta_vs_baseline(df: pd.DataFrame, baseline_id: str | None, metric_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in metric_cols:
        out[f'{col}_delta_vs_baseline'] = np.nan
    if out.empty or not baseline_id or baseline_id not in out['requested_model_run_id'].astype(str).tolist():
        return out
    baseline_row = out[out['requested_model_run_id'].astype(str) == str(baseline_id)].iloc[0]
    for col in metric_cols:
        if col in out.columns and col in baseline_row.index:
            out[f'{col}_delta_vs_baseline'] = out[col] - baseline_row[col]
    return out


def build_replay_command(project_root: Path, historical_csv: Path, weather_csv: Path | None, holidays_xlsx: Path | None, artifacts_root: Path, requested_model_ids: list[str]) -> list[str]:
    command = [
        sys.executable,
        str(project_root / 'scripts' / 'benchmark_replay_models.py'),
        '--historical-csv', str(historical_csv),
        '--start-date', str(START_DATE),
        '--end-date', str(END_DATE),
        '--artifacts-root', str(artifacts_root),
        '--device', str(EXECUTION_DEVICE),
    ]
    if weather_csv is not None:
        command.extend(['--weather-csv', str(weather_csv)])
    if holidays_xlsx is not None:
        command.extend(['--holidays-xlsx', str(holidays_xlsx)])
    if ALLOW_FALLBACK:
        command.append('--allow-fallback')
    command.extend(str(run_id) for run_id in requested_model_ids)
    return command


def run_replay_benchmark(project_root: Path, command: list[str]) -> tuple[Path, Path | None]:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(project_root / 'src') + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    print('Commande lancee:')
    print(' '.join(shlex.quote(part) for part in command))
    completed = subprocess.run(command, cwd=project_root, env=env, capture_output=True, text=True)
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr.strip():
            print(completed.stderr)
        raise RuntimeError(f'Replay benchmark en erreur code={completed.returncode}')
    payload = json.loads(completed.stdout)
    summary_csv = Path(payload['summary_csv']).resolve()
    manifest_json = Path(payload['manifest_json']).resolve() if payload.get('manifest_json') else None
    return summary_csv, manifest_json


def load_payloads(summary_df: pd.DataFrame, metadata_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    replay_frames = []
    per_day_frames = []
    for _, row in summary_df.iterrows():
        replay_csv = row.get('replay_csv')
        if pd.notna(replay_csv) and replay_csv and Path(replay_csv).exists():
            replay_df = pd.read_csv(replay_csv)
            replay_df['Date'] = pd.to_datetime(replay_df['Date'], errors='coerce')
            replay_df['requested_model_run_id'] = str(row['requested_model_run_id'])
            replay_df['effective_model_run_ids'] = row.get('effective_model_run_ids')
            replay_df['fallback_used'] = row.get('fallback_used', False)
            replay_frames.append(replay_df)
        metrics_json = row.get('metrics_json')
        if pd.notna(metrics_json) and metrics_json and Path(metrics_json).exists():
            payload = load_json(metrics_json)
            day_df = pd.DataFrame(payload.get('per_day_metrics', []))
            if not day_df.empty:
                day_df['requested_model_run_id'] = str(payload.get('requested_model_run_id', row['requested_model_run_id']))
                day_df['fallback_used'] = payload.get('fallback_used', False)
                per_day_frames.append(day_df)
    replay_all_df = pd.concat(replay_frames, ignore_index=True) if replay_frames else pd.DataFrame()
    per_day_df = pd.concat(per_day_frames, ignore_index=True) if per_day_frames else pd.DataFrame()
    replay_all_df = add_labels(replay_all_df, metadata_df) if not replay_all_df.empty else replay_all_df
    per_day_df = add_labels(per_day_df, metadata_df) if not per_day_df.empty else per_day_df
    return replay_all_df, per_day_df


def replay_single_run(project_root: Path, catalog_df: pd.DataFrame, run_id: str, historical_csv: Path, weather_csv: Path | None, holidays_xlsx: Path | None, artifacts_root: Path):
    from smartgrid.inference.day_ahead import build_forecast_runtime, forecast_target_day

    match = catalog_df[catalog_df['run_id'].astype(str) == str(run_id)]
    if match.empty:
        raise FileNotFoundError(f'Run introuvable: {run_id}')
    bundle_dir = Path(match.iloc[0]['bundle_dir'])
    runtime = build_forecast_runtime(
        historical_csv=historical_csv,
        current_dir=bundle_dir,
        artifacts_root=artifacts_root,
        weather_csv=weather_csv,
        holidays_xlsx=holidays_xlsx,
        device_request=EXECUTION_DEVICE,
        allow_fallback=ALLOW_FALLBACK,
        logger=None,
    )
    replay_frames = []
    day_rows = []
    for day in pd.date_range(START_DATE, END_DATE, freq='D'):
        target_date = str(day.date())
        day_forecast = forecast_target_day(runtime, target_date, logger=None)
        replay_frames.append(day_forecast)
        if not day_forecast.empty:
            metrics = compute_metrics(day_forecast['Ptot_TOTAL_Real'], day_forecast['Ptot_TOTAL_Forecast'])
            day_rows.append({'target_date': target_date, **metrics, 'requested_model_run_id': str(run_id), 'fallback_used': False})
    replay_df = pd.concat(replay_frames, ignore_index=True) if replay_frames else pd.DataFrame()
    replay_df['requested_model_run_id'] = str(run_id)
    effective_model_run_ids = sorted(str(x) for x in replay_df['model_run_id'].dropna().unique().tolist()) if not replay_df.empty else [str(run_id)]
    summary_row = {
        'requested_model_run_id': str(run_id),
        'effective_model_run_ids': '|'.join(effective_model_run_ids),
        'fallback_enabled': bool(ALLOW_FALLBACK),
        'fallback_used': any(model_id != str(run_id) for model_id in effective_model_run_ids),
        'start_date': str(START_DATE),
        'end_date': str(END_DATE),
        'n_days': int(len(pd.date_range(START_DATE, END_DATE, freq='D'))),
        'n_rows': int(len(replay_df)),
        'replay_csv': None,
        'metrics_json': None,
        **(compute_metrics(replay_df['Ptot_TOTAL_Real'], replay_df['Ptot_TOTAL_Forecast']) if not replay_df.empty else {'count': 0}),
    }
    return summary_row, replay_df, pd.DataFrame(day_rows)


## 3. Preparation, execution optionnelle et chargement

Cette cellule detecte le projet, resout baseline/current, choisit les modeles, lance eventuellement le replay, puis charge les sorties pour l'analyse.


In [ ]:
PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT).resolve() if MANUAL_PROJECT_ROOT else find_project_root()
ensure_project_on_path(PROJECT_ROOT)

CATALOG_DF = collect_runs(PROJECT_ROOT)
FEATURE_BENCHMARK_CSV = find_feature_benchmark_csv(PROJECT_ROOT)
FEATURE_BENCHMARK_DF = load_feature_benchmark(FEATURE_BENCHMARK_CSV)
CURRENT_RUN_ID_RESOLVED = CURRENT_RUN_ID or current_run_id(PROJECT_ROOT)
BASELINE_RUN_ID_RESOLVED = resolve_baseline_run_id(CATALOG_DF, FEATURE_BENCHMARK_DF)
HISTORICAL_CSV_RESOLVED, WEATHER_CSV_RESOLVED, HOLIDAYS_XLSX_RESOLVED, ARTIFACTS_ROOT_RESOLVED = resolve_inputs(PROJECT_ROOT, CATALOG_DF, CURRENT_RUN_ID_RESOLVED, BASELINE_RUN_ID_RESOLVED)
REQUESTED_MODEL_RUN_IDS_RESOLVED = default_requested_models(CATALOG_DF, FEATURE_BENCHMARK_DF, CURRENT_RUN_ID_RESOLVED, BASELINE_RUN_ID_RESOLVED)

if RUN_REPLAY_BENCHMARK:
    if not REQUESTED_MODEL_RUN_IDS_RESOLVED:
        raise ValueError('Aucun modele selectionne pour le replay benchmark.')
    if HISTORICAL_CSV_RESOLVED is None or not HISTORICAL_CSV_RESOLVED.exists():
        raise FileNotFoundError(f'Historique introuvable: {HISTORICAL_CSV_RESOLVED}')
    command = build_replay_command(PROJECT_ROOT, HISTORICAL_CSV_RESOLVED, WEATHER_CSV_RESOLVED, HOLIDAYS_XLSX_RESOLVED, ARTIFACTS_ROOT_RESOLVED, REQUESTED_MODEL_RUN_IDS_RESOLVED)
    REPLAY_SUMMARY_CSV, REPLAY_MANIFEST_JSON = run_replay_benchmark(PROJECT_ROOT, command)
else:
    if MANUAL_REPLAY_SUMMARY_CSV:
        REPLAY_SUMMARY_CSV = Path(MANUAL_REPLAY_SUMMARY_CSV).resolve()
        REPLAY_MANIFEST_JSON = Path(MANUAL_REPLAY_MANIFEST_JSON).resolve() if MANUAL_REPLAY_MANIFEST_JSON else None
    else:
        REPLAY_SUMMARY_CSV, REPLAY_MANIFEST_JSON = find_latest_replay_benchmark(PROJECT_ROOT)

summary_df = pd.read_csv(REPLAY_SUMMARY_CSV).copy()
summary_df.columns = [str(col).strip() for col in summary_df.columns]
if 'run_id' in summary_df.columns and 'requested_model_run_id' not in summary_df.columns:
    summary_df = summary_df.rename(columns={'run_id': 'requested_model_run_id'})
summary_df['requested_model_run_id'] = summary_df['requested_model_run_id'].astype(str)

metadata_df = build_metadata(summary_df, CATALOG_DF)
summary_df = add_labels(summary_df, metadata_df)
replay_all_df, per_day_df = load_payloads(summary_df, metadata_df)

required_refs = []
if COMPARE_WITH_BASELINE and BASELINE_RUN_ID_RESOLVED:
    required_refs.append(BASELINE_RUN_ID_RESOLVED)
if COMPARE_WITH_CURRENT and CURRENT_RUN_ID_RESOLVED:
    required_refs.append(CURRENT_RUN_ID_RESOLVED)

available_runs = set(summary_df['requested_model_run_id'].astype(str).tolist())
for run_id in required_refs:
    if not run_id or run_id in available_runs:
        continue
    if HISTORICAL_CSV_RESOLVED is None or not HISTORICAL_CSV_RESOLVED.exists():
        continue
    summary_row, replay_df_local, per_day_df_local = replay_single_run(PROJECT_ROOT, CATALOG_DF, run_id, HISTORICAL_CSV_RESOLVED, WEATHER_CSV_RESOLVED, HOLIDAYS_XLSX_RESOLVED, ARTIFACTS_ROOT_RESOLVED)
    summary_df = pd.concat([summary_df, pd.DataFrame([summary_row])], ignore_index=True)
    replay_all_df = pd.concat([replay_all_df, replay_df_local], ignore_index=True) if not replay_all_df.empty else replay_df_local.copy()
    if not per_day_df_local.empty:
        per_day_df = pd.concat([per_day_df, per_day_df_local], ignore_index=True) if not per_day_df.empty else per_day_df_local.copy()
    available_runs.add(str(run_id))

metadata_df = build_metadata(summary_df, CATALOG_DF)
summary_df = add_labels(summary_df, metadata_df)
replay_all_df = add_labels(replay_all_df, metadata_df) if not replay_all_df.empty else replay_all_df
per_day_df = add_labels(per_day_df, metadata_df) if not per_day_df.empty else per_day_df
summary_df = delta_vs_baseline(summary_df, BASELINE_RUN_ID_RESOLVED, [col for col in ['MAE', 'RMSE'] if col in summary_df.columns])
summary_df = summary_df.sort_values([col for col in ['MAE', 'RMSE'] if col in summary_df.columns], ascending=True).reset_index(drop=True)

if SELECTED_MODELS is None:
    default_models = [run_id for run_id in REQUESTED_MODEL_RUN_IDS_RESOLVED if run_id in summary_df['requested_model_run_id'].astype(str).tolist()]
    if not default_models:
        default_models = summary_df['requested_model_run_id'].astype(str).head(DISPLAY_TOP_ROWS).tolist()
    SELECTED_MODELS = default_models
else:
    SELECTED_MODELS = [str(run_id) for run_id in SELECTED_MODELS]

selected_summary_df = summary_df[summary_df['requested_model_run_id'].isin(SELECTED_MODELS)].copy()
selected_replay_df = replay_all_df[replay_all_df['requested_model_run_id'].isin(SELECTED_MODELS)].copy()
selected_per_day_df = per_day_df[per_day_df['requested_model_run_id'].isin(SELECTED_MODELS)].copy()

if SELECTED_DAY is None and not selected_per_day_df.empty:
    if CURRENT_RUN_ID_RESOLVED and CURRENT_RUN_ID_RESOLVED in selected_per_day_df['requested_model_run_id'].astype(str).tolist() and 'MAE' in selected_per_day_df.columns:
        current_day_df = selected_per_day_df[selected_per_day_df['requested_model_run_id'].astype(str) == str(CURRENT_RUN_ID_RESOLVED)].copy()
        SELECTED_DAY = str(current_day_df.sort_values('MAE', ascending=False).iloc[0]['target_date'])
    else:
        SELECTED_DAY = str(selected_per_day_df['target_date'].astype(str).iloc[-1])

NOTEBOOK_DATA_CONFIG = resolve_notebook_data_config(PROJECT_ROOT)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('NOTEBOOK_DATA_CONFIG =', {key: NOTEBOOK_DATA_CONFIG.get(key) for key in ['dataset_key', 'historical_csv', 'benchmark_csv', 'weather_csv', 'holidays_xlsx']})
print('FEATURE_BENCHMARK_CSV =', FEATURE_BENCHMARK_CSV)
print('CURRENT_RUN_ID =', CURRENT_RUN_ID_RESOLVED)
print('BASELINE_RUN_ID =', BASELINE_RUN_ID_RESOLVED)
print('REPLAY_SUMMARY_CSV =', REPLAY_SUMMARY_CSV)
print('REPLAY_MANIFEST_JSON =', REPLAY_MANIFEST_JSON)
print('HISTORICAL_CSV =', HISTORICAL_CSV_RESOLVED)
print('WEATHER_CSV =', WEATHER_CSV_RESOLVED)
print('HOLIDAYS_XLSX =', HOLIDAYS_XLSX_RESOLVED)
print('REQUESTED_MODEL_RUN_IDS =', REQUESTED_MODEL_RUN_IDS_RESOLVED)
print('SELECTED_MODELS =', SELECTED_MODELS)
print('SELECTED_DAY =', SELECTED_DAY)
print('summary_df.shape =', summary_df.shape)
print('replay_all_df.shape =', replay_all_df.shape)
print('per_day_df.shape =', per_day_df.shape)

catalog_cols = [col for col in ['run_id', 'config_name', 'display_name', 'n_features', 'arch_label', 'RMSE_train_benchmark', 'MAE_train_benchmark', 'is_baseline_config'] if col in CATALOG_DF.columns]
CATALOG_DF[catalog_cols].head(DISPLAY_TOP_ROWS)


## 4. Classement global des modeles


In [ ]:
ranking_cols = [
    col for col in [
        'model_label', 'config_name', 'feature_signature', 'arch_label', 'n_features',
        'requested_model_run_id', 'effective_model_run_ids', 'fallback_used',
        'MAE', 'MAE_delta_vs_baseline', 'RMSE', 'RMSE_delta_vs_baseline',
        'MAPE%', 'SMAPE%', 'InTolerance%', 'P95AbsError'
    ] if col in summary_df.columns
]
summary_df[ranking_cols].head(DISPLAY_TOP_ROWS)


In [ ]:
plot_df = selected_summary_df.copy()
metric_for_sort = 'RMSE' if 'RMSE' in plot_df.columns else 'MAE'
plot_df = plot_df.dropna(subset=[metric_for_sort]).sort_values(metric_for_sort, ascending=True)
if plot_df.empty:
    print('Aucune donnee a tracer pour le classement replay.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    axes[0].barh(plot_df['model_label'], plot_df['MAE'])
    axes[0].set_title('MAE replay par modele')
    axes[0].invert_yaxis()
    axes[1].barh(plot_df['model_label'], plot_df['RMSE'])
    axes[1].set_title('RMSE replay par modele')
    axes[1].invert_yaxis()
    plt.tight_layout()
    plt.show()


## 5. Analyse jour par jour


In [ ]:
if selected_per_day_df.empty:
    print('Aucune metrique par jour disponible.')
else:
    per_day_pivot = selected_per_day_df.pivot_table(index='target_date', columns='model_label', values='MAE')
    print(per_day_pivot)

    plt.figure(figsize=(16, 6))
    for model_label, group in selected_per_day_df.groupby('model_label'):
        group = group.sort_values('target_date')
        plt.plot(group['target_date'], group['MAE'], marker='o', label=model_label)
    plt.title('MAE par jour sur la plage de replay')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

    if len(per_day_pivot.columns) > 1:
        daily_winners = per_day_pivot.idxmin(axis=1).value_counts().rename_axis('model_label').reset_index(name='n_best_days')
        print(daily_winners)


## 6. Vue globale sur toute la periode


In [ ]:
plot_df = selected_replay_df.copy()
if not plot_df.empty and len(plot_df['Date'].dropna().unique()) > PLOT_LAST_N_POINTS:
    keep_dates = sorted(plot_df['Date'].dropna().unique())[-PLOT_LAST_N_POINTS:]
    plot_df = plot_df[plot_df['Date'].isin(keep_dates)].copy()

if plot_df.empty:
    print('Aucune donnee replay a tracer.')
else:
    plt.figure(figsize=(18, 6))
    real_drawn = False
    for model_label, group in plot_df.groupby('model_label'):
        group = group.sort_values('Date')
        plt.plot(group['Date'], group['Ptot_TOTAL_Forecast'], label=model_label)
        if not real_drawn and 'Ptot_TOTAL_Real' in group.columns and group['Ptot_TOTAL_Real'].notna().any():
            plt.plot(group['Date'], group['Ptot_TOTAL_Real'], label='REAL', color='black', linewidth=2)
            real_drawn = True
    plt.title('Replay - vue globale sur la periode selectionnee')
    plt.legend()
    plt.tight_layout()
    plt.show()

comparison_rows = []
for model_id, group in selected_replay_df.groupby('requested_model_run_id'):
    row = {'requested_model_run_id': model_id, 'model_label': group['model_label'].iloc[0]}
    row.update(compute_metrics(group['Ptot_TOTAL_Real'], group['Ptot_TOTAL_Forecast']))
    comparison_rows.append(row)
comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    comparison_df = delta_vs_baseline(comparison_df, BASELINE_RUN_ID_RESOLVED, [col for col in ['MAE', 'RMSE'] if col in comparison_df.columns])
    comparison_df = comparison_df.sort_values([col for col in ['MAE', 'RMSE'] if col in comparison_df.columns], ascending=True).reset_index(drop=True)
print(comparison_df)

if not selected_replay_df.empty:
    err_box_df = selected_replay_df.copy()
    err_box_df['abs_err'] = (err_box_df['Ptot_TOTAL_Forecast'] - err_box_df['Ptot_TOTAL_Real']).abs()
    err_box_df = err_box_df.pivot_table(index='Date', columns='model_label', values='abs_err')
    err_box_df.plot(kind='box', figsize=(15, 5), title='Distribution des erreurs absolues sur le replay')
    plt.tight_layout()
    plt.show()


## 7. Zoom sur une journee


In [ ]:
if not SELECTED_DAY:
    print('Aucun jour selectionne.')
    day_df = pd.DataFrame()
else:
    day_df = selected_replay_df[selected_replay_df['target_date'].astype(str) == str(SELECTED_DAY)].copy().sort_values('Date')

print(day_df[[col for col in ['Date', 'model_label', 'Ptot_TOTAL_Forecast', 'Ptot_TOTAL_Real'] if col in day_df.columns]].head() if not day_df.empty else day_df)

if day_df.empty:
    print('Aucune donnee disponible pour la journee selectionnee.')
else:
    fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
    real_drawn = False
    for model_label, group in day_df.groupby('model_label'):
        axes[0].plot(group['Date'], group['Ptot_TOTAL_Forecast'], label=model_label, linewidth=2)
        if not real_drawn and group['Ptot_TOTAL_Real'].notna().any():
            axes[0].plot(group['Date'], group['Ptot_TOTAL_Real'], label='REAL', color='black', linewidth=2)
            real_drawn = True
        axes[1].plot(group['Date'], (group['Ptot_TOTAL_Forecast'] - group['Ptot_TOTAL_Real']).abs(), label=model_label)
    axes[0].set_title(f'Jour d analyse {SELECTED_DAY}')
    axes[1].set_title(f'Erreur absolue intra-journaliere - {SELECTED_DAY}')
    axes[0].legend()
    axes[1].legend()
    plt.tight_layout()
    plt.show()

day_rows = []
for model_id, group in day_df.groupby('requested_model_run_id'):
    row = {'requested_model_run_id': model_id, 'model_label': group['model_label'].iloc[0]}
    row.update(compute_metrics(group['Ptot_TOTAL_Real'], group['Ptot_TOTAL_Forecast']))
    day_rows.append(row)
day_metrics_df = pd.DataFrame(day_rows)
if not day_metrics_df.empty:
    day_metrics_df = delta_vs_baseline(day_metrics_df, BASELINE_RUN_ID_RESOLVED, [col for col in ['MAE', 'RMSE'] if col in day_metrics_df.columns])
    day_metrics_df = day_metrics_df.sort_values([col for col in ['MAE', 'RMSE'] if col in day_metrics_df.columns], ascending=True).reset_index(drop=True)
print(day_metrics_df)


## 8. Profils d'erreur et fallback


In [ ]:
if selected_replay_df.empty:
    print('Aucune donnee pour les profils d erreur.')
else:
    work = selected_replay_df.copy()
    work['hour'] = pd.to_datetime(work['Date']).dt.hour
    work['date_only'] = pd.to_datetime(work['Date']).dt.date
    work['abs_err'] = (work['Ptot_TOTAL_Forecast'] - work['Ptot_TOTAL_Real']).abs()

    hourly = work.groupby(['hour', 'model_label'])['abs_err'].mean().reset_index()
    daily = work.groupby(['date_only', 'model_label'])['abs_err'].mean().reset_index()

    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    for model_label, group in hourly.groupby('model_label'):
        axes[0].plot(group['hour'], group['abs_err'], marker='o', label=model_label)
    axes[0].set_title('Erreur absolue moyenne par heure')
    axes[0].legend()

    for model_label, group in daily.groupby('model_label'):
        axes[1].plot(group['date_only'], group['abs_err'], marker='o', label=model_label)
    axes[1].set_title('Erreur absolue moyenne par jour')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

fallback_cols = [col for col in ['model_label', 'config_name', 'requested_model_run_id', 'effective_model_run_ids', 'fallback_enabled', 'fallback_used', 'MAE', 'RMSE'] if col in summary_df.columns]
print(summary_df[fallback_cols])
